In [2]:
base_dir = "C:\\Users\\liamt\\OneDrive - american.edu\\School\\NSAI\\Data"

In [3]:
import pandas as pd

# Load the dataset
# Ensure the file is in the same directory as your script/notebook
df = pd.read_csv(f'{base_dir}\\METABRIC.csv')

# Basic inspection
print(f"Shape of dataset: {df.shape}")
df.info()
print(df.head())

Shape of dataset: (2509, 40)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 2509 entries, 0 to 2508
Data columns (total 40 columns):
 #   Column                          Non-Null Count  Dtype  
---  ------                          --------------  -----  
 0   Patient ID                      2509 non-null   object 
 1   Age at Diagnosis                2498 non-null   float64
 2   Type of Breast Surgery          1955 non-null   object 
 3   Cancer Type                     2509 non-null   object 
 4   Cancer Type Detailed            2509 non-null   object 
 5   Cellularity                     1917 non-null   object 
 6   Chemotherapy                    1980 non-null   object 
 7   Pam50 + Claudin-low subtype     1980 non-null   object 
 8   Cohort                          2498 non-null   float64
 9   ER status measured by IHC       2426 non-null   object 
 10  ER Status                       2469 non-null   object 
 11  G-Neoplasm Histologic Grade     2388 non-null   float64
 12  HER2 

In [4]:
patients, features = df.shape
print(f"Number of Patients: {patients}")
print(f"Number of Features: {features}")

Number of Patients: 2509
Number of Features: 40


In [5]:
# Check unique values and counts for the target variable
balance = df['Relapse Free Status'].value_counts()
percentage = df['Relapse Free Status'].value_counts(normalize=True) * 100

print("Class Balance:")
print(balance)
print("\nPercentage:")
print(percentage)

Class Balance:
Relapse Free Status
Not Recurred    1486
Recurred        1002
Name: count, dtype: int64

Percentage:
Relapse Free Status
Not Recurred    59.726688
Recurred        40.273312
Name: proportion, dtype: float64


In [6]:
# Separate numerical and categorical features
numerical_features = df.select_dtypes(include=['int64', 'float64']).columns.tolist()
categorical_features = df.select_dtypes(include=['object']).columns.tolist()

print(f"Numerical features ({len(numerical_features)}): {numerical_features}")
print(f"Categorical features ({len(categorical_features)}): {categorical_features}")

Numerical features (13): ['Age at Diagnosis', 'Cohort', 'G-Neoplasm Histologic Grade', 'Lymph nodes examined positive', 'Mutation Count', 'Nottingham prognostic index', 'Overall Survival (Months)', 'Relapse Free Status (Months)', 'Tumor Size(mm)', 'Tumor Stage', 'S-Tumor-Size', 'Survival Status', 'RecurredNotRecurredStatus']
Categorical features (27): ['Patient ID', 'Type of Breast Surgery', 'Cancer Type', 'Cancer Type Detailed', 'Cellularity', 'Chemotherapy', 'Pam50 + Claudin-low subtype', 'ER status measured by IHC', 'ER Status', 'HER2 status measured by SNP6', 'HER2 Status', 'Tumor Other Histologic Subtype', 'Hormone Therapy', 'Inferred Menopausal State', 'Integrative Cluster', 'Primary Tumor Laterality', 'Oncotree Code', 'Overall Survival Status', 'PR Status', 'Radio Therapy', 'Relapse Free Status', 'Sex', '3-Gene classifier subtype', "Patient's Vital Status", 'TNBC', 'NPI-Prognosis', 'Age40']


In [7]:
# Calculate missing values per column
missing_data = df.isnull().sum().sort_values(ascending=False)
missing_percentage = (df.isnull().sum() / len(df) * 100).sort_values(ascending=False)

print("Top 5 features with most missing data:")
print(missing_percentage.head(5))

Top 5 features with most missing data:
3-Gene classifier subtype    29.693105
Tumor Stage                  28.736548
Primary Tumor Laterality     25.468314
Cellularity                  23.595058
Type of Breast Surgery       22.080510
dtype: float64


In [9]:
# Initial try at mirroring data preprocessing from book chapter


# 1. Drop Unnecessary/Repetitive Features
# These were identified as having no importance or being redundant [cite: 244-248, 251]
cols_to_drop = [
    'Patient ID', 'Cohort', 'Sex', 'Cancer Type', 'Cancer Type Detailed',
    'ER status measured by IHC', 'HER2 status measured by SNP6', 
    'Integrative Cluster', 'Oncotree Code', '3-Gene classifier subtype',
    'NPI-Prognosis', 'Nottingham prognostic index' # Repetitive or metadata [cite: 244]
]
df_cleaned = df.drop(columns=cols_to_drop)

# 2. Handle Missing Values
# The chapter recommends dropping rows with missing entries to avoid skewing results 
initial_count = len(df_cleaned)
df_cleaned = df_cleaned.dropna()
dropped_count = initial_count - len(df_cleaned)
print(f"Dropped {dropped_count} rows containing missing values.")

# 3. Address Data Leakage (Cautionary Step)
# The chapter included 'Relapse Free Status (Months)' in the final features[cite: 505, 720].
# In prospective modeling, using "time-to-event" columns is considered data leakage.
# If you want to strictly follow the paper, leave them. If you want a robust model, drop them:
leaky_cols = ['Relapse Free Status (Months)', 'Overall Survival (Months)', 'Overall Survival Status', "Patient's Vital Status"]
df_final = df_cleaned.drop(columns=leaky_cols)

# 4. Binary Encoding for Target
# Ensuring the target 'Relapse Free Status' is numerical for the models [cite: 252-253]
df_final['Relapse Free Status'] = df_final['Relapse Free Status'].map({'Not Recurred': 0, 'Recurred': 1})

print("Cleaning complete. Final features:", df_final.columns.tolist())

Dropped 1269 rows containing missing values.
Cleaning complete. Final features: ['Age at Diagnosis', 'Type of Breast Surgery', 'Cellularity', 'Chemotherapy', 'Pam50 + Claudin-low subtype', 'ER Status', 'G-Neoplasm Histologic Grade', 'HER2 Status', 'Tumor Other Histologic Subtype', 'Hormone Therapy', 'Inferred Menopausal State', 'Primary Tumor Laterality', 'Lymph nodes examined positive', 'Mutation Count', 'PR Status', 'Radio Therapy', 'Relapse Free Status', 'Tumor Size(mm)', 'Tumor Stage', 'TNBC', 'S-Tumor-Size', 'Survival Status', 'RecurredNotRecurredStatus', 'Age40']


In [10]:
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

# 1. Define the specific features used in the chapter's best model [cite: 503-508]
target = 'Relapse Free Status'
features = [
    'ER Status', 'PR Status', 'Radio Therapy', 
    'Tumor Size(mm)', 'Pam50 + Claudin-low subtype', 
    'Relapse Free Status (Months)'
]

# Ensure categorical variables are encoded as the paper suggests [cite: 252-253]
X = pd.get_dummies(df_cleaned[features], drop_first=True)
y = df_cleaned[target].map({'Not Recurred': 0, 'Recurred': 1})

# 2. Split the data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# 3. Initialize the Gradient Boosting Classifier with the professor's optimized parameters 
model = GradientBoostingClassifier(
    learning_rate=0.01,
    max_depth=3,
    n_estimators=200
)

# 4. Train and Evaluate
model.fit(X_train, y_train)
y_pred = model.predict(X_test)

print("--- Performance Metrics ---")
print(classification_report(y_test, y_pred))

--- Performance Metrics ---
              precision    recall  f1-score   support

           0       0.73      0.89      0.80       132
           1       0.83      0.62      0.71       116

    accuracy                           0.76       248
   macro avg       0.78      0.75      0.75       248
weighted avg       0.77      0.76      0.76       248



In [11]:
import pandas as pd
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

# 1. Reload to recover the full patient count
df = pd.read_csv(f'{base_dir}\\METABRIC.csv')


# 2. Define the exact features used in the professor's winning model [cite: 503-508, 720]
target = 'Relapse Free Status'
model_features = [
    'ER Status', 
    'PR Status', 
    'Radio Therapy', 
    'Tumor Size(mm)', 
    'Pam50 + Claudin-low subtype', 
    'Relapse Free Status (Months)'
]

# 3. Targeted Cleaning: Drop rows ONLY if one of these 6 columns is missing [cite: 165, 241-242]
# This prevents losing patients who have missing data in columns we aren't even using
df_model = df.dropna(subset=model_features + [target])

# 4. Encoding [cite: 252-253]
X = pd.get_dummies(df_model[model_features], drop_first=True)
y = df_model[target].map({'Not Recurred': 0, 'Recurred': 1})

# 5. Split and Train with Optimized Hyperparameters [cite: 817]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

prof_model = GradientBoostingClassifier(
    learning_rate=0.01,
    max_depth=3,
    n_estimators=200
)

prof_model.fit(X_train, y_train)
y_pred = prof_model.predict(X_test)

# 6. Final Check
print(f"New Training Support: {len(df_model)} patients")
print(classification_report(y_test, y_pred))

New Training Support: 1953 patients
              precision    recall  f1-score   support

           0       0.81      0.86      0.83       243
           1       0.74      0.66      0.70       148

    accuracy                           0.79       391
   macro avg       0.77      0.76      0.77       391
weighted avg       0.78      0.79      0.78       391

